# 02. Testy sparametryzowane

## Cele

Po tym notebooku potrafisz:

* zamienić wiele niemal identycznych testów na jeden test sparametryzowany (`TEST_P`),
* przekazać do testu tabelę przypadków (`Values`, `ValuesIn`) i kombinacje parametrów (`Combine`),
* nadać przypadkom czytelne nazwy i nauczyć GoogleTest wypisywać własne typy.

## Problem: kopiuj-wklej

W notebooku 1 tabela ocen miała dziesięć wartości brzegowych. Gdyby każdą sprawdzał osobny test,
otrzymalibyśmy dziesięć funkcji różniących się tylko liczbami:

```cpp
TEST(GradeTest, Points49) { EXPECT_EQ(grade(49), 2); }
TEST(GradeTest, Points50) { EXPECT_EQ(grade(50), 3); }
TEST(GradeTest, Points69) { EXPECT_EQ(grade(69), 3); }
// ... and seven more
```

Taki kod trudno czytać i utrzymywać: zmiana sposobu sprawdzania wymaga poprawienia każdej kopii,
a nowy przypadek to kolejna kopia. Z kolei upchnięcie wszystkich asercji w jednym teście sprawia,
że raport mówi tylko „test nie przeszedł”, a nie *który przypadek*.

**Testy sparametryzowane wartościami** (*value-parameterized tests*) rozwiązują oba problemy: logikę
testu piszemy raz, a dane podajemy osobno, jako tabelę. Każdy wiersz tabeli staje się osobnym testem
([Advanced — Value-Parameterized Tests](https://google.github.io/googletest/advanced.html#value-parameterized-tests)).

## `TEST_P` krok po kroku

1. **Klasa testowa** dziedzicząca po `::testing::TestWithParam<T>`, gdzie `T` to typ parametru.
   Często jest to prosta struktura opisująca jeden przypadek.
2. **`TEST_P(Klasa, NazwaTestu)`** — treść testu; bieżący parametr zwraca `GetParam()`.
3. **`INSTANTIATE_TEST_SUITE_P(Prefiks, Klasa, generator)`** — podaje wartości parametru.
   Prefiks odróżnia od siebie kilka zestawów danych dla tej samej klasy.

```cpp
#include <stdexcept>

#include <gtest/gtest.h>

int grade(int points) {
    if (points < 0 || points > 100) {
        throw std::out_of_range("points must be in [0, 100]");
    }
    if (points < 50) return 2;
    if (points < 70) return 3;
    if (points < 90) return 4;
    return 5;
}

struct GradeCase {
    int points;
    int expected;
};

class GradeTest : public ::testing::TestWithParam<GradeCase> {};

TEST_P(GradeTest, MapsPointsToGrade) {
    const GradeCase& c = GetParam();
    EXPECT_EQ(grade(c.points), c.expected);
}

INSTANTIATE_TEST_SUITE_P(
    Boundaries, GradeTest,
    ::testing::Values(GradeCase{0, 2}, GradeCase{49, 2},
                      GradeCase{50, 3}, GradeCase{69, 3},
                      GradeCase{70, 4}, GradeCase{89, 4},
                      GradeCase{90, 5}, GradeCase{100, 5}));

// Invalid input is a different behaviour (an exception), so it gets its own test.
class InvalidPointsTest : public ::testing::TestWithParam<int> {};

TEST_P(InvalidPointsTest, Throws) {
    EXPECT_THROW(grade(GetParam()), std::out_of_range);
}

INSTANTIATE_TEST_SUITE_P(OutOfRange, InvalidPointsTest,
                         ::testing::Values(-1, 101, -1000, 1000));
```

GoogleTest nazywa wygenerowane testy według schematu `Prefiks/Klasa.NazwaTestu/indeks`, np.
`Boundaries/GradeTest.MapsPointsToGrade/3`. Liczba testów to iloczyn: (liczba `TEST_P` w klasie)
× (liczba wartości) dla każdego `INSTANTIATE_TEST_SUITE_P`.

Zwróć uwagę, że przypadki niepoprawne mają osobny test. Jeden `TEST_P` powinien sprawdzać jedno
zachowanie — gdy w tabeli pojawia się kolumna „czy ma rzucić wyjątek” i `if` w treści testu,
to znak, że należy go podzielić.

## Generatory parametrów

Trzeci argument `INSTANTIATE_TEST_SUITE_P` to **generator**
([Testing Reference — parameter generators](https://google.github.io/googletest/reference/testing.html#param-generators)):

| Generator | Daje |
|---|---|
| `Values(v1, v2, ..., vN)` | podane wartości |
| `ValuesIn(kontener)` | elementy tablicy, kontenera STL lub zakresu iteratorów — wygodne, gdy tabela jest zdefiniowana osobno |
| `Range(begin, end, step)` | `begin`, `begin + step`, ... — **bez** `end` |
| `Bool()` | `false`, `true` |
| `Combine(g1, ..., gN)` | wszystkie kombinacje (iloczyn kartezjański) jako `std::tuple` |

```cpp
#include <tuple>
#include <vector>

#include <gtest/gtest.h>

int clampToByte(int value) {
    if (value < 0) return 0;
    if (value > 255) return 255;
    return value;
}

// ValuesIn: the table lives in an ordinary container.
const std::vector<int> kInRange{0, 1, 128, 254, 255};

class ClampInRangeTest : public ::testing::TestWithParam<int> {};

TEST_P(ClampInRangeTest, KeepsValue) {
    EXPECT_EQ(clampToByte(GetParam()), GetParam());
}

INSTANTIATE_TEST_SUITE_P(Table, ClampInRangeTest, ::testing::ValuesIn(kInRange));

// Combine: every pair (a, b) from two lists - 3 x 2 = 6 tests.
class ClampSumTest : public ::testing::TestWithParam<std::tuple<int, int>> {};

TEST_P(ClampSumTest, ResultIsAByte) {
    const auto [a, b] = GetParam();
    const int result = clampToByte(a + b);
    EXPECT_GE(result, 0);
    EXPECT_LE(result, 255);
}

INSTANTIATE_TEST_SUITE_P(Pairs, ClampSumTest,
                         ::testing::Combine(::testing::Values(-300, 0, 300),
                                            ::testing::Values(-1, 1)));
```

Więcej przykładów: [sample7 (podstawy) i sample8 (`Combine`)](https://google.github.io/googletest/samples.html)
w repozytorium GoogleTest.

## Czytelne nazwy przypadków

Nazwa `.../3` nie mówi, który przypadek zawiódł. Opcjonalny czwarty argument
`INSTANTIATE_TEST_SUITE_P` to funkcja, która tworzy nazwę z parametru
([Advanced — Specifying Names](https://google.github.io/googletest/advanced.html#specifying-names-for-value-parameterized-test-parameters)).
Dostaje `::testing::TestParamInfo<T>` z polami `param` (wartość) i `index` (numer) i zwraca `std::string`.

```cpp
INSTANTIATE_TEST_SUITE_P(
    Boundaries, GradeTest,
    ::testing::Values(GradeCase{0, 2}, GradeCase{49, 2} /* ... */),
    [](const ::testing::TestParamInfo<GradeCase>& info) {
        return "Points" + std::to_string(info.param.points);
    });
```

Zasady:

* nazwa może zawierać **tylko litery, cyfry i podkreślnik** — znaki takie jak `-`, spacja czy `.`
  są niedozwolone (uwaga na liczby ujemne!),
* nazwy w obrębie jednej instancji muszą być unikalne,
* gotowy generator `::testing::PrintToStringParamName` działa dla prostych typów (np. `int`),
  ale **nie działa dla `std::string` ani napisów C** — dla nich pisze się własną lambdę.

Gdy przypadki nie mają naturalnej nazwy, często dodaje się do struktury przypadku pole `name`
i zwraca po prostu `info.param.name`.

## Czytelne komunikaty: wypisywanie własnych typów

Gdy test sparametryzowany nie przechodzi, GoogleTest wypisuje wartość parametru. Dla własnej
struktury nie wie jednak jak, więc pokazuje jej surowe bajty. Wystarczy zdefiniować funkcję `PrintTo`
(albo `operator<<`) **w tej samej przestrzeni nazw co typ**
([Advanced — Teaching GoogleTest How to Print Your Values](https://google.github.io/googletest/advanced.html#teaching-googletest-how-to-print-your-values)):

```cpp
void PrintTo(const GradeCase& c, std::ostream* os) {
    *os << "{points: " << c.points << ", expected: " << c.expected << "}";
}
```

To samo dotyczy zwykłych asercji: `EXPECT_EQ(date1, date2)` dla własnego typu `Date` wypisze
czytelne wartości tylko wtedy, gdy GoogleTest umie je wydrukować.

## Demonstracja

Plik [`demos/grade_test.cpp`](demos/grade_test.cpp) zawiera tabelę ocen jako test sparametryzowany —
z generatorem nazw i funkcją `PrintTo`. Komórka poniżej buduje go, wypisuje listę testów
(`ctest -N`) i uruchamia je.

Uwaga: CMake (`gtest_discover_tests`) w nazwach testów CTest domyślnie zastępuje indeks parametru
jego wartością, dlatego dla `InvalidPointsTest` zobaczysz np. `.../-1`, a nie `.../0`. Sam GoogleTest
używa indeksów ([CMake — GoogleTest](https://cmake.org/cmake/help/latest/module/GoogleTest.html),
opcja `NO_PRETTY_VALUES`).

In [ ]:
# Build the demo, list the generated tests and run them
!cmake -S demos -B demos/build && cmake --build demos/build --config Debug --parallel && ctest --test-dir demos/build -C Debug -N && ctest --test-dir demos/build -C Debug --output-on-failure

Spróbuj: zmień w pliku oczekiwaną ocenę w jednym wierszu tabeli, zapisz plik i uruchom komórkę ponownie.
Zobacz, jak wygląda komunikat o błędzie — nazwa przypadku i wartość parametru wskazują dokładnie,
który wiersz tabeli jest zły.

## Ćwiczenie

*Treść ćwiczenia — w przygotowaniu.*

## Linki i materiały

* [Advanced — Value-Parameterized Tests](https://google.github.io/googletest/advanced.html#value-parameterized-tests)
* [Testing Reference — `TEST_P`, `INSTANTIATE_TEST_SUITE_P`, generatory](https://google.github.io/googletest/reference/testing.html#INSTANTIATE_TEST_SUITE_P)
* [Advanced — Teaching GoogleTest How to Print Your Values](https://google.github.io/googletest/advanced.html#teaching-googletest-how-to-print-your-values)
* [GoogleTest Samples](https://google.github.io/googletest/samples.html) — przykłady 7 i 8

---

[← 01. Podstawy GoogleTest](../01_basics/01_basics.ipynb) | [Strona główna](../_start.ipynb) | [03. Fixture'y i cykl życia testu →](../03_fixtures/03_fixtures.ipynb)